# NumPy: Interactive Tour

The array library everything else (`torch`, `pandas`) copies its API from.
Run every cell. Some ask you to predict the output *before* running, which
is the actual retention technique, not just decoration.


```python
import numpy as np
```


In [ ]:
# type it yourself, then run

## Arrays: creation and shape

An array's two most important properties are `.shape` (a tuple, one entry
per dimension) and `.dtype` (what kind of number it stores, this matters
more than in plain Python, since NumPy is statically typed under the hood
for speed).


```python
a = np.array([1, 2, 3])
b = np.array([[1, 2, 3], [4, 5, 6]])
print(a.shape, a.dtype)
print(b.shape, b.dtype)

print(np.zeros((2, 3)))
print(np.arange(0, 10, 2))       # start, stop, step
print(np.linspace(0, 1, 5))      # start, stop, N evenly spaced points
```


In [ ]:
# type it yourself, then run

**Predict before running**: what shape does `np.zeros((3, 1, 4))` have,
and how many total numbers does it hold?


```python
z = np.zeros((3, 1, 4))
print(z.shape, z.size)
```


In [ ]:
# type it yourself, then run

## Indexing and slicing

Same `[start:stop:step]` syntax as Python lists, but extended across
multiple dimensions with commas, and it extends to **boolean masks**: an
array of `True`/`False` the same shape as what you're indexing, keeping only
the `True` positions. Boolean masking is the single most-used NumPy/PyTorch
idiom; get comfortable with it here.


```python
m = np.arange(12).reshape(3, 4)
print(m)
print("row 1:       ", m[1])
print("column 2:     ", m[:, 2])
print("bottom-right: ", m[1:, 2:])

mask = m % 2 == 0
print("mask:\n", mask)
print("even values: ", m[mask])   # always returns a flat 1D array
```


In [ ]:
# type it yourself, then run

## Broadcasting

NumPy lets you combine arrays of *different* shapes without manually
looping, as long as their shapes are "compatible": matching from the right,
where a dimension of size 1 stretches to match. This is the mechanism behind
`resid_pre + attn_out` style additions you saw in the mech interp notebooks,
except there it's implicit inside PyTorch instead of NumPy.


```python
row = np.array([1, 2, 3, 4])         # shape (4,)
col = np.array([[10], [20], [30]])   # shape (3, 1)
print(row + col)                     # broadcasts to shape (3, 4)
```


In [ ]:
# type it yourself, then run

**Predict before running**: what shape does `np.ones((5, 1, 3)) + np.ones((1, 4, 3))` produce?


```python
result = np.ones((5, 1, 3)) + np.ones((1, 4, 3))
print(result.shape)
```


In [ ]:
# type it yourself, then run

## Reductions and the `axis` argument

`.sum()`, `.mean()`, `.max()`, `.argmax()` collapse an array down, and
`axis` controls *which* dimension gets collapsed. This trips everyone up at
first: `axis=0` collapses dimension 0 (so the result loses that dimension,
keeping the others), not "sum along axis 0's direction" in some other sense.


```python
m = np.arange(12).reshape(3, 4)
print(m)
print("sum all:        ", m.sum())
print("sum axis=0 (per column):", m.sum(axis=0))   # shape (4,), collapsed the 3 rows
print("sum axis=1 (per row):   ", m.sum(axis=1))   # shape (3,), collapsed the 4 columns
print("argmax axis=1:          ", m.argmax(axis=1))
```


In [ ]:
# type it yourself, then run

## Reshaping

`.reshape()` changes shape without changing the data or its order (it's
still the same flat sequence of numbers underneath). `.T` (or `.transpose()`)
actually reorders data, it swaps axes. `np.newaxis` (or `None`) inserts a
size-1 dimension, useful for making broadcasting line up on purpose.


```python
v = np.arange(6)
print(v.reshape(2, 3))
print(v.reshape(2, 3).T)             # transpose: shape becomes (3, 2)
print(v[:, np.newaxis].shape)        # (6,) -> (6, 1)
```


In [ ]:
# type it yourself, then run

## Combining arrays: `concatenate` vs `stack`

`np.concatenate([a, b], axis=k)` joins arrays along an *existing* axis (they
must already match on every other axis). `np.stack([a, b], axis=k)` joins
along a *new* axis, so the inputs must be exactly the same shape and the
result gains one extra dimension. This is the numpy analog of
`torch.cat(...)`, used constantly to build batches out of repeated pieces.


```python
a = np.zeros((2, 3))
b = np.ones((2, 3))
print(np.concatenate([a, b], axis=0).shape)  # (4, 3), no new dim
print(np.stack([a, b], axis=0).shape)        # (2, 2, 3), new dim at front
```


In [ ]:
# type it yourself, then run

## `einsum`: naming axes instead of remembering `@`/`.T` conventions

`np.einsum(subscripts, *arrays)` describes a tensor contraction by naming
each array's axes with letters and writing which letters survive into the
output. It's more verbose than `@` for a plain matrix multiply, but it makes
batched, multi-axis operations (like attention score computation) legible
instead of a pile of `.transpose()` calls.


```python
a = np.random.rand(3, 4)
b = np.random.rand(4, 5)
print(np.array_equal(np.einsum('ij,jk->ik', a, b).round(6), (a @ b).round(6)))

# batched, multi-head attention scores:
# [batch, head, q_pos, d] x [batch, head, k_pos, d] -> [batch, head, q_pos, k_pos]
q = np.random.rand(2, 4, 6, 8)
k = np.random.rand(2, 4, 6, 8)
scores = np.einsum('bhqd,bhkd->bhqk', q, k)
print(scores.shape)
```


In [ ]:
# type it yourself, then run

## Reducing over more than one axis at once

`axis` accepts a tuple, collapsing several dimensions in one call. Handy
when you want to average over a batch dimension and some other axis without
reshaping first.


```python
x = np.random.rand(2, 4, 6)
print(x.mean(axis=(0, -1)).shape)  # collapses axis 0 AND axis -1 -> (4,)
```


In [ ]:
# type it yourself, then run

## `np.diagonal()` on a 3D array

`np.diagonal(a, offset=k, axis1=, axis2=)` extracts a (optionally shifted)
diagonal from two of an array's axes, leaving any other axes alone. This is
the exact mechanic behind scoring "attends to the position right after the
one it's copying" in a batch of attention patterns shaped
`[batch, dest_pos, src_pos]`.


```python
pattern = np.random.rand(3, 5, 5)  # [batch, dest_pos, src_pos]
diag = np.diagonal(pattern, offset=-1, axis1=-2, axis2=-1)
print(diag.shape)  # diagonal moves to the last axis: [batch, diag_len]
```


In [ ]:
# type it yourself, then run

## Reproducible random numbers

`np.random.default_rng(seed)` gives you a generator object. Seed it once,
reuse it, and every call is reproducible. Prefer this over the legacy
`np.random.seed(...)` global-state API.


```python
rng = np.random.default_rng(0)
print(rng.random(3))
print(rng.integers(0, 10, size=3))
```


In [ ]:
# type it yourself, then run

## Checkpoint: write a few lines yourself

Given `pattern` below (a `[batch=2, dest=4, src=4]` array standing in for a
toy attention pattern), compute `score`: the mean of the values one step
below the main diagonal (`offset=-1`) between the last two axes, averaged
over the batch too, a single scalar. Use `np.diagonal(pattern, offset=-1,
axis1=-2, axis2=-1)` then `.mean()`.

In [ ]:
pattern = np.array([
    [[0.1, 0.2, 0.3, 0.4],
     [0.5, 0.6, 0.7, 0.8],
     [0.9, 1.0, 1.1, 1.2],
     [1.3, 1.4, 1.5, 1.6]],
    [[2.1, 2.2, 2.3, 2.4],
     [2.5, 2.6, 2.7, 2.8],
     [2.9, 3.0, 3.1, 3.2],
     [3.3, 3.4, 3.5, 3.6]],
])

score = None  # YOUR CODE HERE

In [ ]:
# self-check
assert score is not None, "fill in `score` above"
_expected = np.diagonal(pattern, offset=-1, axis1=-2, axis2=-1).mean()
assert np.isclose(score, _expected), f"got {score}, expected {_expected}"
print("Checkpoint passed")

## Cheat sheet

| Want to... | Call |
|---|---|
| Make an array | `np.array(list)`, `np.zeros(shape)`, `np.ones(shape)`, `np.arange(start, stop, step)` |
| Shape / dtype | `.shape`, `.dtype`, `.size` (total element count) |
| Slice | `a[1:3]`, `a[:, 0]`, `a[mask]` where `mask` is a boolean array |
| Combine different shapes | broadcasting, matches shapes from the right, size-1 dims stretch |
| Collapse a dimension | `a.sum(axis=k)`, `a.mean(axis=k)`, `a.argmax(axis=k)` |
| Collapse several dimensions at once | `a.mean(axis=(0, -1))` |
| Change shape, same data | `a.reshape(new_shape)` |
| Reorder axes | `a.T`, `a.transpose(order)` |
| Insert a size-1 axis | `a[:, np.newaxis]` |
| Join along an existing axis | `np.concatenate([a, b], axis=k)` |
| Join along a brand-new axis | `np.stack([a, b], axis=k)` |
| Named-axis tensor contraction | `np.einsum('ij,jk->ik', a, b)` |
| Extract a (shifted) diagonal | `np.diagonal(a, offset=k, axis1=, axis2=)` |
| Random numbers | `rng = np.random.default_rng(seed); rng.random(shape)` |